# DermaFair — Quickstart

This notebook demonstrates the **portable fairness engine** on synthetic predictions, so you can see the outputs before wiring up the full DermaCon-IN training pipeline.

The engine is framework-agnostic: it operates on prediction arrays + a group vector, so you can drop it onto *any* classifier.

In [ ]:
import numpy as np
import pandas as pd
from dermafair.fairness import FairnessEvaluator
from dermafair.visualization import (
    fairness_heatmap, accuracy_fairness_pareto,
    gate_weights_by_tone, fusion_comparison, compile_report,
)
from pathlib import Path

rng = np.random.default_rng(0)
out = Path('demo_outputs'); out.mkdir(exist_ok=True)

## 1. Simulate predictions for seven models
Darker Fitzpatrick bands receive more errors for less-fair models.

In [ ]:
ev = FairnessEvaluator('fitzpatrick')
model_bias = {
    'cnn': 0.35, 'resnet50': 0.18, 'vit_b16': 0.20, 'hybrid': 0.12,
    'metadata': 0.40, 'late_fusion': 0.15, 'gate_network': 0.04,
}
rows = []
for name, dark_penalty in model_bias.items():
    fitz = rng.choice([3,4,5,6], 300)
    y = rng.integers(0,2,300); yp = y.copy()
    for k,b in enumerate(fitz):
        if rng.random() < 0.08 + dark_penalty*(b-3)/3:
            yp[k] = 1-yp[k]
    rows.append(ev.evaluate(y, yp, fitz, n_bootstrap=300).to_row(name))
master = pd.DataFrame(rows)
master.round(3)

## 2. Generate the publication figures

In [ ]:
fairness_heatmap(master, out/'fairness_heatmap.png')
accuracy_fairness_pareto(master, out/'accuracy_fairness_pareto.png')
fusion_comparison(master, out/'fusion_comparison.png')
print('figures written to', out)

## 3. The signature analysis: gate weighting by skin tone
Simulate a gate that leans on metadata more as tone darkens, then visualize.

In [ ]:
fitz = rng.choice([3,4,5,6], 300)
# image weight decreases with darker tone (lower image-lesion contrast)
w_image = np.clip(0.75 - 0.10*(fitz-3) + rng.normal(0,0.05,300), 0, 1)
gw = pd.DataFrame({'fitzpatrick': fitz, 'w_image': w_image, 'w_metadata': 1-w_image})
gate_weights_by_tone(gw, out/'gate_weights_by_tone.png')
corr = np.corrcoef(gw['fitzpatrick'], gw['w_metadata'])[0,1]
print(f'Corr(Fitzpatrick band, metadata weight) = {corr:.3f}')

## 4. Auto-compile the narrative report

In [ ]:
compile_report(master, out/'fairness_report.md', 'demo')
print(open(out/'fairness_report.md').read())

## Next steps
Wire up the real pipeline:
```bash
python scripts/prepare_data.py --config configs/dermacon.yaml
python scripts/train_all.py    --config configs/dermacon.yaml
python scripts/run_fairness.py --config configs/dermacon.yaml
```